In [5]:
from pdf2docx import Converter
from pathlib import Path
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("pdf_test")

def test_pdf_conversion(input_pdf_path: str, output_folder: str):
    pdf_path = Path(input_pdf_path.strip())
    
    target_dir = Path(output_folder)
    target_dir.mkdir(parents=True, exist_ok=True) 
    
    docx_path = target_dir / f"{pdf_path.stem}.docx"

    cv = Converter(str(pdf_path))
    cv.convert(str(docx_path), start=0, end=None)
    cv.close()

    return docx_path
        

In [6]:
pdf_path = "../infra/CUAD_v1/full_contract_pdf/Part_II/Supply/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.PDF"
docx_path = test_pdf_conversion(pdf_path, "../infra/docx")

[INFO] Start to convert ../infra/CUAD_v1/full_contract_pdf/Part_II/Supply/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.PDF
[INFO] [1/4] Opening document...
[INFO] [2/4] Analyzing document...
[WARNING] Ignore Line "character of that information as Confidential Information hereunder" due to overlap
[INFO] [3/4] Parsing pages...
[INFO] (1/54) Page 1
[INFO] (2/54) Page 2
[INFO] (3/54) Page 3
[INFO] (4/54) Page 4
[INFO] (5/54) Page 5
[INFO] (6/54) Page 6
[INFO] (7/54) Page 7
[INFO] (8/54) Page 8
[INFO] (9/54) Page 9
[INFO] (10/54) Page 10
[INFO] (11/54) Page 11
[INFO] (12/54) Page 12
[INFO] (13/54) Page 13
[INFO] (14/54) Page 14
[INFO] (15/54) Page 15
[INFO] (16/54) Page 16
[INFO] (17/54) Page 17
[INFO] (18/54) Page 18
[INFO] (19/54) Page 19
[INFO] (20/54) Page 20
[INFO] (21/54) Page 21
[INFO] (22/54) Page 22
[INFO] (23/54) Page 23
[INFO] (24/54) Page 24
[INFO] (25/54) Page 25
[INFO] (26/54) Page 26
[INFO] (27/54) Page 27
[INFO] (28/54) Page 28
[INFO] (29/54) Page 29
[INF

In [9]:
docx_path

PosixPath('../infra/docx/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.docx')

In [11]:
import mammoth

def docx_to_html(docx_path):
    with open(docx_path, "rb") as docx_file:
        result = mammoth.convert_to_html(docx_file)
        html = result.value
        return html

html = docx_to_html(docx_path)
html

'<p><strong>Exhibit 10.1 </strong><br /><strong>[***] = Certain confidential information contained in this document, </strong><br /><strong>marked by brackets, has been omitted because it is both </strong><br /><strong>(i) not material and (ii) would likely be competitively harmful if publicly disclosed.</strong></p><p>Miltenyi Biotec-Bellicum <br />Supply Agreement <br />(Execution Copy March 27, 2019)</p><p><strong>SUPPLY AGREEMENT</strong></p><p>(MB Global Contract Number MBGCR 19001)</p><p>This Supply Agreement (this “Agreement”) is made and entered into, effective as of March 27, 2019 (the “Effective Date”), by and between Miltenyi Biotec GmbH, a German corporation having an address at Friedrich-Ebert-Str. 68, 51429 Bergisch Gladbach, Germany (hereinafter referred to as “Miltenyi”), and Bellicum Pharmaceuticals, Inc., a US corporation, having a registered office at 2130 West Holcombe Boulevard, Suite 800, Houston, TX 77030 (on behalf of itself and its Affiliates, individually and 

In [15]:
from docx import Document

def docx_to_json(docx_path):
    doc = Document(docx_path)
    paragraphs = []

    for i, para in enumerate(doc.paragraphs):
        paragraphs.append({
            "id": f"p{i}",
            "text": para.text,
            "style": para.style
        })

    return paragraphs

paragraphs = docx_to_json(docx_path)
paragraphs

[{'id': 'p0',
  'text': '',
  'style': _ParagraphStyle('Normal') id: 139961098844816},
 {'id': 'p1',
  'text': 'Exhibit 10.1 \n[***] = Certain confidential information contained in this document, \nmarked by brackets, has been omitted because it is both \n(i) not material and (ii) would likely be competitively harmful if publicly disclosed.',
  'style': _ParagraphStyle('Normal') id: 139961098843984},
 {'id': 'p2',
  'text': 'Miltenyi Biotec-Bellicum \nSupply Agreement \n(Execution Copy March 27, 2019)',
  'style': _ParagraphStyle('Normal') id: 139961098840016},
 {'id': 'p3',
  'text': 'SUPPLY AGREEMENT',
  'style': _ParagraphStyle('Normal') id: 139961098839760},
 {'id': 'p4',
  'text': '(MB Global Contract Number MBGCR 19001)',
  'style': _ParagraphStyle('Normal') id: 139961098830736},
 {'id': 'p5',
  'text': 'This Supply Agreement (this “Agreement”) is made and entered into, effective as of March 27, 2019 (the “Effective Date”), by and between Miltenyi Biotec GmbH, a German corporatio

In [16]:
import fitz

def pdf_to_structured_json(pdf_path):
    doc = fitz.open(pdf_path)
    output = []
    pid = 0

    for page_index, page in enumerate(doc):
        data = page.get_text("dict")

        for block in data["blocks"]:
            if block["type"] != 0:
                continue

            paragraph = {
                "id": f"p{pid}",
                "page": page_index,
                "bbox": block["bbox"],
                "runs": []
            }

            for line in block["lines"]:
                for span in line["spans"]:
                    paragraph["runs"].append({
                        "text": span["text"],
                        "size": span["size"],
                        "font": span["font"],
                        "bold": "Bold" in span["font"],
                        "italic": "Italic" in span["font"],
                        "color": span["color"]
                    })

            output.append(paragraph)
            pid += 1

    return output

output = pdf_to_structured_json(pdf_path)
output


[{'id': 'p0',
  'page': 0,
  'bbox': (317.70001220703125,
   51.24300003051758,
   544.9500732421875,
   67.65589141845703),
  'runs': [{'text': 'Exhibit 10.1',
    'size': 7.575000286102295,
    'font': 'TimesNewRomanPS-BoldMT',
    'bold': True,
    'italic': False,
    'color': 0},
   {'text': '[***] = Certain confidential information contained in this document,',
    'size': 7.575000286102295,
    'font': 'TimesNewRomanPS-BoldMT',
    'bold': True,
    'italic': False,
    'color': 0}]},
 {'id': 'p1',
  'page': 0,
  'bbox': (272.25, 68.91877746582031, 544.9500122070312, 85.33167266845703),
  'runs': [{'text': 'marked by brackets, has been omitted because it is both',
    'size': 7.575000286102295,
    'font': 'TimesNewRomanPS-BoldMT',
    'bold': True,
    'italic': False,
    'color': 0},
   {'text': '(i) not material and (ii) would likely be competitively harmful if publicly disclosed.',
    'size': 7.575000286102295,
    'font': 'TimesNewRomanPS-BoldMT',
    'bold': True,
    'i